In [ ]:
# install first:  pip install pypdf

from pypdf import PdfReader

def extract_pdf_text(filepath):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        extracted_text=page.extract_text()
        
        text+=extracted_text+"\n"
        
    return text

full_text = extract_pdf_text("Mukhlif_Aljaberi_CV (2).pdf")
import re
full_text = re.sub(r'\s+', ' ', full_text)  
print(f"Total characters: {len(full_text)}")
print(full_text[:500])  

Total characters: 2037
Computer Engineering graduate specialising in Artificial Intelligence, combining strong academic foundations in deep learning, computer vision and generative AI with hands-on experience delivering full-stack AI-powered web applications. Proven ability to research and implement cutting-edge models (GANs, diffusion, face-swap) within production pipelines. Eager to contribute to innovative AI teams and continue growing as an engineer. Mukhlif Mohammed Aljaberi AI & Computer Engineering Graduate mok


In [2]:
def chunk_text_from_string(text, chunk_size=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        x = ' '.join(chunk)
        chunks.append(x)
    return chunks

chunks = chunk_text_from_string(full_text)
print(f"Got {len(chunks)} chunks")

Got 2 chunks


In [3]:
# a new collection for the CV-vs-job tool
import chromadb

# creates a persistent database saved to disk in a folder called "chroma_db"
chroma_client = chromadb.PersistentClient(path="./chroma_db_cv_job")
cv_job_collection = chroma_client.get_or_create_collection(name="cv_job_matcher")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # small, free, good enough

def embed_chunks(chunks):
    embeddings = []
    for chunk in chunks:
        embidding=model.encode(chunk)
        embeddings.append(embidding)
        pass
    return embeddings

embeddings = embed_chunks(chunks)
print(len(embeddings))        
print(len(embeddings[0]))     

c:\Users\Lenovo\Desktop\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8605.67it/s]


2
384


In [ ]:
def add_document(text, source_label, collection):
    # clean + chunk 
    import re
    text = re.sub(r'\s+', ' ', text)
    doc_chunks = chunk_text_from_string(text, chunk_size=150)
    
    # embed
    doc_embeddings = [emb.tolist() for emb in embed_chunks(doc_chunks)]
    
    ids = [f"{source_label}_chunk_{i}" for i in range(len(doc_chunks))]
    
    metadatas =[{"source":source_label} for _ in range(len(doc_chunks))]
    
    collection.add(
        documents=doc_chunks,
        embeddings=doc_embeddings,
        ids=ids,
        metadatas=metadatas
    )
    print(f"Added {len(doc_chunks)} chunks from '{source_label}'")

In [ ]:
def retrieve_for_matching(query, collection, top_k=3):
    query_embedding = model.encode(query).tolist()
    
    # retrieve top chunks ONLY from the CV
    cv_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"source": "cv"}    
    )
    
    job_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"source": "job"}
    )
    
    cv_chunks = cv_results['documents'][0]
    job_chunks = job_results['documents'][0]
    return cv_chunks, job_chunks
    
cv_chunks, job_chunks = retrieve_for_matching("what technical skills are required?", cv_job_collection)
print("FROM CV:", cv_chunks)
print("\nFROM JOB:", job_chunks)

FROM CV: ['Computer Engineering graduate specialising in Artificial Intelligence, combining strong academic foundations in deep learning, computer vision and generative AI with hands-on experience delivering full-stack AI-powered web applications. Proven ability to research and implement cutting-edge models (GANs, diffusion, face-swap) within production pipelines. Eager to contribute to innovative AI teams and continue growing as an engineer. Mukhlif Mohammed Aljaberi AI & Computer Engineering Graduate mokhlifaljabri2001@gmail.com +963 986 961 587 EDUCATION B.Sc. Computer Engineering — Specialisation in Artificial Intelligence Arab International University Graduation Year: 2026 GPA: 2.7 / 4.0 TECHNICAL SKILLS AI / ML / DL Machine Learning Deep Learning GANs Computer Vision PyTorch TensorFlow AI MODELS & TOOLS YOLO / YOLOe InsightFace SDXL TinyLlama In-painting Stable Diffusion LANGUAGES Python SQL HTML / CSS JavaScript FRAMEWORKS Laravel React Bootstrap ASP .NET ACADEMIC PROJECTS AI Fu

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
client = Groq()  
from groq import Groq

client = Groq() 

def analyze_match(cv_chunks, job_chunks):
    cv_context = "\n".join(cv_chunks)
    job_context = "\n".join(job_chunks)
    
    prompt = f"""You are a career advisor. Compare the candidate's CV against the job requirements below and give an honest assessment.

CANDIDATE'S CV (relevant parts):
{cv_context}

JOB REQUIREMENTS (relevant parts):
{job_context}

Provide:
1. MATCH: What does the candidate genuinely have that this role needs?
2. GAPS: What key requirements does the candidate NOT appear to meet?
3. HONEST VERDICT: Is this a strong fit, a stretch, or a mismatch? Be direct, not encouraging for its own sake.

Base your answer ONLY on what's in the CV and job text above."""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=700
    )
    return response.choices[0].message.content

query = "what are the key skills and requirements for this role?"
cv_chunks, job_chunks = retrieve_for_matching(query, cv_job_collection)
print(analyze_match(cv_chunks, job_chunks))

**MATCH:**
The candidate has some technical skills that align with the job requirements, such as:
- Experience with Machine Learning (listed as a technical skill)
- Proficiency in Python (a required technical skill)
- Familiarity with data processing and modeling (demonstrated through academic projects)

**GAPS:**
The candidate appears to lack:
- Direct experience with statistical or econometric modeling (e.g., regression, regularization techniques, model validation)
- Experience with specific tools like R, EViews, SAS, or cloud environments (Azure, AWS, GCP)
- Hands-on experience with data visualization tools like Power BI or Tableau
- Explicit experience with data ingestion, processing, and modeling in a professional setting (most experience listed is from academic projects)

**HONEST VERDICT:**
This is a mismatch. While the candidate has a strong foundation in AI, Machine Learning, and programming, their experience and skills do not directly align with the specific requirements of t

In [10]:
def analyze_cv_vs_job(cv_file, job_text):
    # 1. extract CV text from the uploaded PDF
    cv_text = extract_pdf_text(cv_file)
    
    # 2. fresh collection each run, so old data doesn't pile up
    #    (delete + recreate to keep it clean)
    try:
        chroma_client.delete_collection(name="cv_job_matcher")
    except:
        pass
    collection = chroma_client.get_or_create_collection(name="cv_job_matcher")
    
    # 3. load both documents with source tags
    add_document(cv_text, "cv", collection)
    add_document(job_text, "job", collection)
    
    # 4. retrieve + analyze
    query = "what are the key skills and requirements for this role?"
    cv_chunks, job_chunks = retrieve_for_matching(query, collection)
    result = analyze_match(cv_chunks, job_chunks)
    
    return result

In [11]:
analyze_cv_vs_job("Mukhlif_Aljaberi_CV (2).pdf",job_text)

Added 2 chunks from 'cv'
Added 8 chunks from 'job'


"**1. MATCH:** \nThe candidate has experience with Machine Learning, which is highly valued in the job requirements. They also have hands-on experience with Python, which is one of the tools mentioned in the job requirements. Additionally, the candidate has worked with large datasets and has experience with data processing and modeling, which are essential skills for the role.\n\n**2. GAPS:** \nThe candidate lacks direct experience with statistical or econometric modeling, which is a key requirement for the role. They also do not mention experience with model validation, diagnostics, and performance metrics, which are crucial skills for the position. Furthermore, the candidate does not appear to have experience with data visualization tools like Power BI or Tableau, or with cloud environments and Git-based version control, which are nice-to-have skills. The candidate's background is more focused on Computer Vision and Generative AI, which may not be directly applicable to the role.\n\n

In [12]:
import gradio as gr

def gradio_analyze(cv_file, job_text):
    if cv_file is None:
        return "Please upload a CV (PDF)."
    if not job_text or job_text.strip() == "":
        return "Please paste a job description."
    return analyze_cv_vs_job(cv_file, job_text)

demo = gr.Interface(
    fn=gradio_analyze,
    inputs=[
        gr.File(label="Upload your CV (PDF)", file_types=[".pdf"], type="filepath"),
        gr.Textbox(label="Paste the job description", lines=12, placeholder="Paste the full job posting here...")
    ],
    outputs=gr.Markdown(label="Match Analysis"),
    title="CV ↔ Job Match Analyzer",
    description="Upload your CV and paste a job description. Get an honest assessment of your fit, your strengths, and your gaps — powered by RAG."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Lenovo\Desktop\RAG\.venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Added 2 chunks from 'cv'
Added 8 chunks from 'job'


In [ ]:
!pip install gradio

In [ ]:
print("ehloo")